# BirdCLEF+ 2026 ONNX Perch + SED Blend Proxy6

Controlled proxy-mapping variant after the 0.890 ONNX Perch + SED champion. This notebook keeps exact-mapped classes unchanged, leaves anonymous insect sonotypes on SED only, and adds a small same-genus Perch proxy contribution for named unmapped taxa when Perch metadata provides candidate labels.

## 1. Setup And Configuration

Use fixed Kaggle input roots and attached wheel/model datasets. This variant keeps the protected champion's exact-mapped blend weight and adds a smaller proxy weight only for named unmapped taxa.

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys
import time
from types import ModuleType
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


class CFG:
    """Configuration for ONNX SED plus ONNX Perch blending."""

    seed = 42
    data_root = Path("/kaggle/input/competitions/birdclef-2026")
    sed_model_dir = Path(
        "/kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public"
    )
    perch_onnx_root = Path(
        "/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx"
    )
    perch_tf_model_dir = Path(
        "/kaggle/input/models/google/"
        "bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"
    )
    output_path = Path("/kaggle/working/submission.csv")

    sample_rate = 32_000
    clip_seconds = 5
    soundscape_seconds = 60
    n_windows = soundscape_seconds // clip_seconds
    samples_per_window = sample_rate * clip_seconds
    samples_per_file = sample_rate * soundscape_seconds

    sed_n_mels = 256
    sed_n_fft = 2048
    sed_hop_length = 512
    sed_fmin = 20
    sed_fmax = 16_000
    sed_top_db = 80
    sed_smooth_sigma = 0.75
    sed_fold_pattern = "sed_fold*.onnx"

    perch_model_patterns = ("perch_v2_no_dft*.onnx", "perch_v2*.onnx")
    onnx_wheel_pattern = "onnxruntime-*.whl"
    onnx_threads = 4
    batch_files = 4
    perch_weight = 0.15
    proxy_weight = 0.05


np.random.seed(CFG.seed)
print("Competition root:", CFG.data_root)
print("SED model root:", CFG.sed_model_dir)
print("Perch ONNX root:", CFG.perch_onnx_root)


In [ ]:
def install_onnxruntime_if_needed() -> ModuleType:
    """Import ONNX Runtime, installing an attached wheel if needed.

    Returns:
        ModuleType: Imported `onnxruntime` module.

    Raises:
        FileNotFoundError: If ONNX Runtime is unavailable and no offline wheel
            is attached under `/kaggle/input`.
    """
    try:
        import onnxruntime as ort

        return ort
    except ImportError:
        wheels = sorted(CFG.perch_onnx_root.glob(CFG.onnx_wheel_pattern))
        if not wheels:
            wheels = sorted(Path("/kaggle/input").glob("**/onnxruntime-*.whl"))
        if not wheels:
            raise FileNotFoundError(
                "No offline onnxruntime wheel found under /kaggle/input."
            )

        wheel = wheels[0]
        print("Installing ONNX Runtime from", wheel)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                str(wheel),
            ],
            check=True,
        )

        import onnxruntime as ort

        return ort


ort = install_onnxruntime_if_needed()
import librosa
import soundfile as sf
from scipy.ndimage import gaussian_filter1d

print("ONNX Runtime:", ort.__version__)


## 2. Submission Contract And Label Mapping

The SED model already predicts the 234 submission columns. ONNX Perch predicts 14,795 global labels, so this notebook uses exact scientific-name matching before any blend is attempted.

In [ ]:
sample_submission = pd.read_csv(CFG.data_root / "sample_submission.csv")
taxonomy = pd.read_csv(CFG.data_root / "taxonomy.csv")
target_columns = [c for c in sample_submission.columns if c != "row_id"]
num_classes = len(target_columns)
label_to_pos = {label: idx for idx, label in enumerate(target_columns)}

print("Sample submission:", sample_submission.shape)
print("Taxonomy:", taxonomy.shape)
print("Target columns:", num_classes)


def find_perch_labels_path() -> Path:
    """Find Perch `labels.csv` metadata from attached Kaggle inputs.

    Returns:
        Path: CSV containing Perch label metadata.

    Raises:
        FileNotFoundError: If no compatible Perch label metadata is found.
    """
    preferred = CFG.perch_tf_model_dir / "assets" / "labels.csv"
    if preferred.exists():
        return preferred

    for path in sorted(Path("/kaggle/input").glob("**/labels.csv")):
        try:
            columns = set(pd.read_csv(path, nrows=0).columns)
        except Exception:
            continue
        if {"inat2024_fsd50k", "scientific_name", "label", "name"} & columns:
            return path

    raise FileNotFoundError(
        "Perch labels.csv not found. Attach the Google Perch model or another "
        "input containing compatible Perch label metadata."
    )


def load_perch_labels(path: Path) -> pd.DataFrame:
    """Load Perch labels and normalize the scientific-name column.

    Args:
        path (Path): Perch label metadata CSV.

    Returns:
        pd.DataFrame: Perch labels with `bc_index` and `scientific_name`.
    """
    labels = (
        pd.read_csv(path)
        .reset_index()
        .rename(
            columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}
        )
    )
    if "scientific_name" not in labels.columns:
        for column in ["label", "labels", "name"]:
            if column in labels.columns:
                labels = labels.rename(columns={column: "scientific_name"})
                break
    if "scientific_name" not in labels.columns:
        raise ValueError(f"No scientific-name column found in {path}")
    labels["scientific_name"] = labels["scientific_name"].astype(str)
    return labels[["bc_index", "scientific_name"]].copy()


def build_exact_perch_mapping(perch_labels: pd.DataFrame) -> pd.DataFrame:
    """Map BirdCLEF submission labels to exact Perch label indices.

    Args:
        perch_labels (pd.DataFrame): Normalized Perch label metadata.

    Returns:
        pd.DataFrame: Mapping rows for all submission labels.
    """
    mapping = taxonomy[["primary_label", "scientific_name"]].merge(
        perch_labels, on="scientific_name", how="left"
    )
    mapping = mapping[mapping["primary_label"].isin(target_columns)].copy()
    mapping["target_pos"] = mapping["primary_label"].map(label_to_pos)
    mapping["is_mapped"] = mapping["bc_index"].notna()
    mapping["bc_index"] = mapping["bc_index"].fillna(-1).astype(int)
    mapping["target_pos"] = mapping["target_pos"].astype(int)
    return mapping.sort_values("target_pos").reset_index(drop=True)


def build_named_genus_proxy_map(
    mapping: pd.DataFrame, perch_labels: pd.DataFrame
) -> dict[int, np.ndarray]:
    """Build same-genus proxy candidates for named unmapped targets only.

    Anonymous insect sonotypes are intentionally skipped because their labels do
    not contain a real genus/species name.

    Args:
        mapping (pd.DataFrame): Exact mapping table for submission columns.
        perch_labels (pd.DataFrame): Normalized Perch label metadata.

    Returns:
        dict[int, np.ndarray]: Target-column position to Perch label indices.
    """
    proxy_map = {}
    unmapped = mapping[~mapping["is_mapped"]].copy()
    perch_names = perch_labels["scientific_name"].astype(str)

    for _, row in unmapped.iterrows():
        scientific_name = str(row["scientific_name"])
        if scientific_name.startswith("Insect son"):
            continue
        parts = scientific_name.split()
        if len(parts) < 2:
            continue

        genus = parts[0]
        hits = perch_labels[perch_names.str.match(rf"^{genus}\s", na=False)]
        if len(hits) == 0:
            continue
        proxy_map[int(row["target_pos"])] = hits["bc_index"].to_numpy(
            dtype=np.int32
        )
    return proxy_map


perch_labels_path = find_perch_labels_path()
perch_labels = load_perch_labels(perch_labels_path)
perch_mapping = build_exact_perch_mapping(perch_labels)
proxy_map = build_named_genus_proxy_map(perch_mapping, perch_labels)

mapped = perch_mapping[perch_mapping["is_mapped"]]
unmapped = perch_mapping[~perch_mapping["is_mapped"]]
sonotype_unmapped = unmapped[
    unmapped["scientific_name"].astype(str).str.startswith("Insect son")
]

mapped_target_pos = mapped["target_pos"].to_numpy(dtype=np.int32)
mapped_perch_idx = mapped["bc_index"].to_numpy(dtype=np.int32)
proxy_target_pos = np.array(sorted(proxy_map), dtype=np.int32)

print("Perch labels:", perch_labels.shape, perch_labels_path)
print(f"Exact Perch mapping: {len(mapped)} / {num_classes}")
print(f"Unmapped target columns: {len(unmapped)}")
print(f"Skipped insect sonotypes: {len(sonotype_unmapped)}")
print(f"Named genus proxies: {len(proxy_map)}")
for target_pos in proxy_target_pos:
    label = target_columns[target_pos]
    n_proxy = len(proxy_map[int(target_pos)])
    print(f" - {label}: {n_proxy} same-genus Perch labels")


## 3. Load ONNX Models

Both model families run with `CPUExecutionProvider`. SED has multiple folds; Perch uses one no-DFT ONNX model.

In [ ]:
def find_sed_folds() -> list[Path]:
    """Find distilled SED ONNX fold files.

    Returns:
        list[Path]: Sorted SED ONNX fold paths.

    Raises:
        FileNotFoundError: If no SED folds are attached.
    """
    folds = sorted(CFG.sed_model_dir.glob(CFG.sed_fold_pattern))
    if not folds:
        folds = sorted(
            Path("/kaggle/input").glob(f"**/{CFG.sed_fold_pattern}")
        )
    if not folds:
        raise FileNotFoundError("No SED ONNX fold files found.")
    return folds


def find_perch_onnx_model() -> Path:
    """Find the ONNX Perch model file.

    Returns:
        Path: ONNX Perch model path.

    Raises:
        FileNotFoundError: If no matching ONNX Perch model is attached.
    """
    for pattern in CFG.perch_model_patterns:
        candidates = sorted(CFG.perch_onnx_root.glob(pattern))
        if not candidates:
            candidates = sorted(Path("/kaggle/input").glob(f"**/{pattern}"))
        if candidates:
            return candidates[0]
    raise FileNotFoundError("No ONNX Perch model found under /kaggle/input.")


def make_session(path: Path) -> ort.InferenceSession:
    """Create an optimized CPU ONNX Runtime session.

    Args:
        path (Path): ONNX model path.

    Returns:
        ort.InferenceSession: Loaded ONNX Runtime CPU session.
    """
    options = ort.SessionOptions()
    options.intra_op_num_threads = CFG.onnx_threads
    options.inter_op_num_threads = 1
    options.graph_optimization_level = (
        ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    )
    return ort.InferenceSession(
        str(path), sess_options=options, providers=["CPUExecutionProvider"]
    )


sed_folds = find_sed_folds()
sed_sessions = [make_session(path) for path in sed_folds]
sed_input_names = [session.get_inputs()[0].name for session in sed_sessions]

perch_model_path = find_perch_onnx_model()
perch_session = make_session(perch_model_path)
perch_input_name = perch_session.get_inputs()[0].name
perch_output_names = [output.name for output in perch_session.get_outputs()]
perch_output_map = {name: idx for idx, name in enumerate(perch_output_names)}

print("SED folds:", len(sed_sessions))
for path in sed_folds:
    print(" -", path)
print("Perch model:", perch_model_path)
print("Perch outputs:", perch_output_names)


## 4. Audio And Prediction Helpers

One 60-second file is loaded once, split into 12 windows, and converted into both model inputs: log-mel tensors for SED and raw waveform windows for Perch.

In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """Apply a numerically stable sigmoid transform.

    Args:
        x (np.ndarray): Logit array.

    Returns:
        np.ndarray: Probability array with the same shape as `x`.
    """
    x = np.clip(x, -40, 40)
    return 1.0 / (1.0 + np.exp(-x))


def load_audio_60s(audio_path: Path) -> np.ndarray:
    """Load one soundscape as a fixed-length mono waveform.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        np.ndarray: Float32 waveform with exactly 60 seconds of audio.
    """
    audio, sr = sf.read(str(audio_path), dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != CFG.sample_rate:
        audio = librosa.resample(
            audio, orig_sr=sr, target_sr=CFG.sample_rate
        ).astype(np.float32)

    if len(audio) < CFG.samples_per_file:
        audio = np.pad(audio, (0, CFG.samples_per_file - len(audio)))
    else:
        audio = audio[: CFG.samples_per_file]
    return audio.astype(np.float32, copy=False)


def waveform_to_windows(audio: np.ndarray) -> np.ndarray:
    """Reshape one 60-second waveform into 5-second windows.

    Args:
        audio (np.ndarray): Fixed-length soundscape waveform.

    Returns:
        np.ndarray: Array shaped `(12, 160000)`.
    """
    return audio.reshape(CFG.n_windows, CFG.samples_per_window)


def windows_to_sed_logmel(windows: np.ndarray) -> np.ndarray:
    """Convert waveform windows into normalized SED log-mel tensors.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        np.ndarray: Float32 tensor shaped `(12, 1, n_mels, frames)`.
    """
    features = []
    for window in windows:
        mel = librosa.feature.melspectrogram(
            y=window,
            sr=CFG.sample_rate,
            n_fft=CFG.sed_n_fft,
            hop_length=CFG.sed_hop_length,
            n_mels=CFG.sed_n_mels,
            fmin=CFG.sed_fmin,
            fmax=CFG.sed_fmax,
            power=2.0,
        )
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=CFG.sed_top_db)
        mel_db = (mel_db + CFG.sed_top_db) / CFG.sed_top_db
        features.append(mel_db.astype(np.float32))
    return np.expand_dims(np.stack(features, axis=0), axis=1)


def predict_sed(windows: np.ndarray) -> np.ndarray:
    """Predict SED probabilities for one soundscape.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        np.ndarray: Probability matrix shaped `(12, 234)`.
    """
    batch = windows_to_sed_logmel(windows)
    logits_sum = np.zeros((CFG.n_windows, num_classes), dtype=np.float32)

    for session, input_name in zip(sed_sessions, sed_input_names):
        outputs = session.run(None, {input_name: batch})
        clip_logits = outputs[0]
        frame_max = outputs[1].max(axis=1)
        logits_sum += 0.5 * clip_logits + 0.5 * frame_max

    logits = logits_sum / len(sed_sessions)
    logits = gaussian_filter1d(
        logits, sigma=CFG.sed_smooth_sigma, axis=0, mode="nearest"
    )
    return sigmoid(logits).astype(np.float32)


def predict_perch_mapped(windows: np.ndarray) -> np.ndarray:
    """Predict exact and proxy-mapped Perch probabilities for one soundscape.

    Args:
        windows (np.ndarray): Raw waveform windows.

    Returns:
        np.ndarray: Probability matrix shaped `(12, 234)`, with zeros for
            columns that have neither exact nor proxy mapping.
    """
    outputs = perch_session.run(None, {perch_input_name: windows})
    if "label" not in perch_output_map:
        raise KeyError(f"Perch output 'label' not found: {perch_output_names}")

    logits = outputs[perch_output_map["label"]].astype(np.float32)
    probs = np.zeros((CFG.n_windows, num_classes), dtype=np.float32)
    probs[:, mapped_target_pos] = sigmoid(logits[:, mapped_perch_idx])

    for target_pos, perch_indices in proxy_map.items():
        proxy_logits = logits[:, perch_indices].max(axis=1)
        probs[:, target_pos] = sigmoid(proxy_logits)
    return probs


def blend_predictions(
    sed_probs: np.ndarray, perch_probs: np.ndarray
) -> np.ndarray:
    """Blend SED, exact Perch, and narrow proxy Perch probabilities.

    Args:
        sed_probs (np.ndarray): SED probabilities shaped `(12, 234)`.
        perch_probs (np.ndarray): Perch probabilities shaped `(12, 234)`.

    Returns:
        np.ndarray: Blended probabilities shaped `(12, 234)`.
    """
    blended = sed_probs.copy()
    w = CFG.perch_weight
    blended[:, mapped_target_pos] = (1.0 - w) * sed_probs[
        :, mapped_target_pos
    ] + w * perch_probs[:, mapped_target_pos]

    if len(proxy_target_pos) > 0:
        proxy_w = CFG.proxy_weight
        blended[:, proxy_target_pos] = (1.0 - proxy_w) * sed_probs[
            :, proxy_target_pos
        ] + proxy_w * perch_probs[:, proxy_target_pos]

    return np.clip(blended, 0.0, 1.0).astype(np.float32)


def predict_file(audio_path: Path) -> tuple[list[str], np.ndarray]:
    """Predict blended probabilities for one soundscape file.

    Args:
        audio_path (Path): Soundscape OGG path.

    Returns:
        tuple[list[str], np.ndarray]: Row IDs and blended probabilities.
    """
    audio = load_audio_60s(audio_path)
    windows = waveform_to_windows(audio)
    sed_probs = predict_sed(windows)
    perch_probs = predict_perch_mapped(windows)
    blended = blend_predictions(sed_probs, perch_probs)

    stem = audio_path.stem
    row_ids = [
        f"{stem}_{(idx + 1) * CFG.clip_seconds}"
        for idx in range(CFG.n_windows)
    ]
    return row_ids, blended


## 5. Write Submission

During public dry runs, Kaggle may expose no `test_soundscapes`; in that case this notebook writes the untouched sample submission. During real scoring, hidden soundscapes are mounted and every expected row must be predicted.

In [ ]:
def list_test_soundscapes() -> list[Path]:
    """List hidden test soundscapes mounted by Kaggle.

    Returns:
        list[Path]: Sorted hidden test OGG paths, or an empty list during
            public dry runs.
    """
    test_dir = CFG.data_root / "test_soundscapes"
    if not test_dir.exists():
        return []
    return sorted(test_dir.glob("*.ogg"))


def build_submission() -> pd.DataFrame:
    """Score hidden soundscapes and write `submission.csv`.

    Returns:
        pd.DataFrame: Submission frame written to `CFG.output_path`.

    Raises:
        ValueError: If hidden scoring expects rows that were not predicted.
    """
    audio_paths = list_test_soundscapes()
    if not audio_paths:
        print("No test soundscapes found. Writing sample submission dry run.")
        submission = sample_submission.copy()
        submission.to_csv(CFG.output_path, index=False)
        return submission

    print(f"Scoring {len(audio_paths)} soundscape files")
    print(f"Exact Perch mapped columns: {len(mapped_target_pos)}")
    print(f"Perch blend weight: {CFG.perch_weight:.2f}")
    print(f"Proxy blend weight: {CFG.proxy_weight:.2f}")
    print(f"Named proxy columns: {len(proxy_map)}")

    start = time.perf_counter()
    rows = []
    values = []

    for idx, audio_path in enumerate(audio_paths, start=1):
        row_ids, probs = predict_file(audio_path)
        rows.extend(row_ids)
        values.append(probs)
        if idx == 1 or idx % 25 == 0 or idx == len(audio_paths):
            elapsed = time.perf_counter() - start
            print(f"{idx}/{len(audio_paths)} files scored in {elapsed:.1f}s")

    pred_df = pd.DataFrame(np.vstack(values), columns=target_columns)
    pred_df.insert(0, "row_id", rows)
    pred_df = pred_df.drop_duplicates("row_id").set_index("row_id")

    missing = sorted(set(sample_submission["row_id"]) - set(pred_df.index))
    if missing:
        raise ValueError(
            f"Missing predictions for {len(missing)} row_ids. "
            f"Example: {missing[:3]}"
        )

    submission = sample_submission[["row_id"]].copy()
    submission[target_columns] = pred_df.loc[
        submission["row_id"], target_columns
    ].to_numpy()
    submission.to_csv(CFG.output_path, index=False)
    print("Saved", CFG.output_path, submission.shape)
    return submission


submission = build_submission()
display(submission.head())
print(submission.shape)
